# CSCI 645 HW2: Imitation learning on PandaPickCube (Diffusion Policy)

One notebook, two platforms. Run it **either** on Google Colab (T4) **or** on a CARC GPU node
(OnDemand JupyterLab with the `Python (hw2)` kernel from `slurm/setup_carc.sh`). Never mix
platforms inside one sweep, because GPU simulation differs slightly across GPU types.

Every cell only shells out to the scripts in `code/`, so the notebook does exactly what the
command-line / SLURM workflow does. Steps are **resumable**: finished results are committed to
GitHub, and re-running a cell skips work that is already done.

**Rules for edits:** change behaviour through `--set key=value` overrides where possible. If you
edit a `.py` file on the runtime, commit + push it (last cell) before the session ends.

In [ ]:
# ---- 0. Settings ----------------------------------------------------------------------------
REPO_URL = "https://github.com/<github-user>/<repo-name>.git"   # private repo (no token here!)
BRANCH = "main"
GIT_NAME, GIT_EMAIL = "kashinadhe", "kaustubhadhe@gmail.com"

import os, sys, subprocess, pathlib, shlex
IN_COLAB = os.path.isdir("/content") and not os.environ.get("SLURM_JOB_ID")
cwd = pathlib.Path.cwd()
if (cwd / "code" / "hw2").is_dir():
    REPO = cwd
elif (cwd.parent / "code" / "hw2").is_dir():
    REPO = cwd.parent                                  # notebook opened from notebooks/ (CARC)
else:
    REPO = pathlib.Path("/content" if IN_COLAB else pathlib.Path.home()) / pathlib.Path(REPO_URL).stem
print("platform:", "Colab" if IN_COLAB else "CARC/other", "| repo:", REPO)

In [ ]:
# ---- 1. GitHub access (token is typed, never saved) + clone / pull ---------------------------
# Use a fine-grained personal access token restricted to this one repo,
# permission "Contents: Read and write" (write is needed to push results back).
from getpass import getpass
if not os.environ.get("GH_TOKEN"):
    os.environ["GH_TOKEN"] = getpass("GitHub fine-grained token: ")
auth_url = REPO_URL.replace("https://", f"https://x-access-token:{os.environ['GH_TOKEN']}@")
os.environ["HW2_PUSH_URL"] = auth_url                  # used by run_sweep.py --push

def git(*args, check=True):
    r = subprocess.run(["git", *args], cwd=REPO if REPO.exists() else None, capture_output=True, text=True)
    out = (r.stdout + r.stderr).replace(os.environ["GH_TOKEN"], "***")
    if out.strip(): print(out.strip())
    if check and r.returncode: raise RuntimeError(f"git {args[0]} failed")
    return r

if not (REPO / ".git").exists():
    r = subprocess.run(["git", "clone", "-q", "-b", BRANCH, auth_url, str(REPO)], capture_output=True, text=True)
    if r.returncode: raise RuntimeError("clone failed: " + r.stderr.replace(os.environ["GH_TOKEN"], "***"))
    git("remote", "set-url", "origin", REPO_URL)       # keep the token out of .git/config
else:
    git("pull", "--ff-only", auth_url, BRANCH)
git("config", "user.name", GIT_NAME); git("config", "user.email", GIT_EMAIL)
os.chdir(REPO)
git("log", "--oneline", "-3")

In [ ]:
# ---- 2. Install (Colab only; CARC uses the venv built by slurm/setup_carc.sh) ----------------
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    # EGL vendor file for headless MuJoCo rendering on Colab GPUs (as in the Playground notebooks).
    icd = "/usr/share/glvnd/egl_vendor.d/10_nvidia.json"
    if not os.path.exists(icd):
        os.makedirs(os.path.dirname(icd), exist_ok=True)
        with open(icd, "w") as f:
            f.write('{"file_format_version" : "1.0.0", "ICD" : {"library_path" : "libEGL_nvidia.so.0"}}')
os.environ["MUJOCO_GL"] = "egl"
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"])

def sh(cmd):
    # Run a pipeline command from the repo root, streaming its output; raise on failure.
    print("$", cmd, flush=True)
    p = subprocess.Popen(shlex.split(cmd), cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout: print(line, end="")
    if p.wait(): raise RuntimeError(f"command failed ({p.returncode}): {cmd}")

def push(paths, message):
    # Commit the given paths (if changed) and push to GitHub.
    paths = [p for p in paths if (REPO / p).exists()]
    if not paths: return print("nothing to push")
    git("add", "-f", *paths)
    if git("diff", "--cached", "--quiet", check=False).returncode:
        git("commit", "-q", "-m", message)
        git("push", "-q", os.environ["HW2_PUSH_URL"], f"HEAD:{BRANCH}")
        print("pushed:", message)
    else:
        print("nothing to commit")

from IPython.display import Image, Video, display, Markdown

## Smoke test
Versions, GPU, environment dimensions, expert success on 50 seeds, a rendered video and
training-speed estimates. **Paste the final `SMOKE TEST REPORT` block back to Claude.**
To tune the expert, re-run with overrides, e.g.
`sh("python code/smoke_test.py --sections env expert render --set expert.v_approach=0.025")`.

In [ ]:
sh("python tests/test_cpu.py")
sh("python code/smoke_test.py")

In [ ]:
for v in ["videos/smoke/expert_smoke.mp4", "videos/smoke/expert_smoke_failure.mp4"]:
    if (REPO / v).exists(): display(Markdown(f"`{v}`"), Video(str(REPO / v), embed=True, width=480))

## Part 1: Collect and inspect demonstrations
Skipped automatically if `data/demos_baseline.npz` is already in the repo (set `FORCE = True` to recollect).

In [ ]:
FORCE = False
if FORCE or not (REPO / "data/demos_baseline.npz").exists():
    sh("python code/collect_demos.py")
sh("python code/inspect_demos.py")
for f in ["fig_frames", "fig_coverage", "fig_trajectories", "fig_actions", "fig_phases"]:
    p = REPO / f"results/demos/{f}.png"
    if p.exists(): display(Image(str(p)))
grid = REPO / "videos/expert/expert_demos_grid.mp4"
if grid.exists(): display(Video(str(grid), embed=True, width=640))
push(["data/demos_baseline.npz", "results/demos", "videos/expert"], "demos: baseline dataset + inspection")

## Teacher evaluation on the held-out evaluation seeds

In [ ]:
if not (REPO / "results/teacher_eval/eval_summary.json").exists():
    sh("python code/evaluate.py --teacher --out-dir results/teacher_eval --video-dir videos/teacher_eval")
push(["results/teacher_eval", "videos/teacher_eval"], "results: teacher evaluation")

## (Optional) Overfitting sanity check
Train on 5 demos and evaluate **on those same 5 seeds**. Near-perfect success here means the
data -> training -> execution path is wired correctly. Not part of the reported results.

In [ ]:
RUN_SANITY = True
if RUN_SANITY:
    sh("python code/train.py --run-dir runs/sanity_n5 --set data.num_demos=5 train.num_steps=3000 eval.batch_size=5")
    sh("python code/evaluate.py --run-dir runs/sanity_n5 --on-train-seeds --out-dir results/sanity_n5 --num-videos 1")

## Parts 2-3: Train + evaluate the dataset-size sweep (resumable)
Runs `configs/sweep.yaml` (100 / 50 / 25 demos x seeds 0, 1, 2). Each finished run is pushed
immediately. If the session dies, reconnect, run cells 0-2, then this cell again.

In [ ]:
sh("python code/run_sweep.py --dry-run")
sh("python code/run_sweep.py --push")

## Part 4: Aggregate, plot, push

In [ ]:
sh("python code/analyze.py")
display(Markdown((REPO / "results/summary_table.md").read_text()))
for f in ["fig_success_vs_demos", "fig_success_vs_threshold", "fig_failure_modes", "fig_loss_curves", "fig_smoothness"]:
    display(Image(str(REPO / f"results/{f}.png")))
push(["results", "videos"], "results: aggregate tables and figures")

In [ ]:
# ---- Commit any code edits made on this runtime (review the diff first!) ---------------------
git("status", "--short")
# push(["code", "configs"], "code: edits made on the runtime")